# Spillover sensitivity of neighbor-harm reward

Notebook 05 found a small pollution reduction and higher mitigation cost with full neighbor-harm reward at spillover 0.2. This exploratory notebook repeats the local-versus-neighbor-harm comparison at spillover 0.1 and 0.4. The values were chosen before these runs. We also show saved 0.0 and 0.2 results. All other environment, training, and evaluation settings stay fixed. Run from the `notebooks` directory.


In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from config import EnvironmentConfig, TrainingConfig
from evaluate import evaluate_policy, summarize_by_policy, summarize_by_training_seed, write_csv
from train import save_agents, train_iql


In [ ]:
train_new_runs = False
write_comparison = False
new_spillovers = (0.1, 0.4)
neighbor_harm_weight = 1.0
training_seeds = (11, 23, 37, 41, 53, 67, 71, 83, 97, 101, 107,
                  109, 127, 131, 137, 149, 157, 163, 173, 179)
evaluation_seeds = tuple(range(100000, 100020))
output_directory = '../results/notebook_runs/spillover_sensitivity_01'
saved_zero_directory = '../results/followup/spillover_00'
saved_original_directory = '../results/followup/spillover_02'
saved_neighbor_directory = '../results/notebook_runs/neighbor_externality_01'

training_config = TrainingConfig()
training_config.training_seeds = training_seeds
training_config.evaluation_seeds = evaluation_seeds

with open(f'{saved_original_directory}/manifest.json', encoding='utf-8') as file:
    reference_manifest = json.load(file)
if tuple(reference_manifest['training']['training_seeds']) != training_seeds:
    raise ValueError('Training seeds differ from the saved comparison.')
if tuple(reference_manifest['training']['evaluation_seeds']) != evaluation_seeds:
    raise ValueError('Evaluation seeds differ from the saved comparison.')
if reference_manifest['training']['training_episodes'] != training_config.training_episodes:
    raise ValueError('Training episode count differs from the saved comparison.')
print('New spillovers:', new_spillovers, 'Neighbor-harm weight:', neighbor_harm_weight)
print('Training seeds:', len(training_seeds), 'Held-out demand seeds:', len(evaluation_seeds))


## Train and evaluate

For each new spillover strength, train local and full neighbor-harm policies with the same 20 training seeds. Each greedy policy faces the same 20 held-out demand sequences. The evaluation metrics use actual pollution and cost, not the training reward. Prepare the two output folders before running this cell. After the first completed run, set `train_new_runs = False` to inspect saved results. Set `write_comparison = True` only when you want to replace the derived comparison tables and figures.


In [ ]:
if train_new_runs:
    for spillover in new_spillovers:
        environment_config = EnvironmentConfig()
        environment_config.spillover = spillover
        condition_directory = f'{output_directory}/spillover_{int(spillover * 100):02d}'
        episode_rows = []
        model_names = []

        for policy_name, reward_mode, weight in (('local', 'local', 0.0),
                                                 ('neighbor_harm', 'externality', neighbor_harm_weight)):
            for seed in training_seeds:
                agents = train_iql(reward_mode, seed, training_config, environment_config, weight)
                model_name = f'q_{policy_name}_{seed}.json'
                save_agents(f'{condition_directory}/{model_name}', agents)
                model_names.append(model_name)

                def greedy_policy(observations, agents=agents):
                    return tuple(agent.select_action(observation, greedy=True)
                                 for agent, observation in zip(agents, observations))

                for metrics in evaluate_policy(greedy_policy, evaluation_seeds, environment_config):
                    episode_rows.append({'policy': policy_name, 'training_seed': seed, **metrics})
                print(f'Trained spillover {spillover}, {policy_name}, seed {seed}', flush=True)

        run_rows = summarize_by_training_seed(episode_rows)
        policy_rows = summarize_by_policy(run_rows)
        write_csv(f'{condition_directory}/episodes.csv', episode_rows)
        write_csv(f'{condition_directory}/run_summary.csv', run_rows)
        write_csv(f'{condition_directory}/policy_summary.csv', policy_rows)
        manifest = {
            'question': 'Local versus neighbor-harm reward across spillover strengths',
            'source': 'notebooks/06_spillover_sensitivity.ipynb',
            'command': 'Run all cells from the notebooks working directory',
            'environment': vars(environment_config),
            'training': vars(training_config),
            'neighbor_harm_weight': neighbor_harm_weight,
            'models': model_names,
            'status': 'Exploratory follow-up',
        }
        with open(f'{condition_directory}/manifest.json', 'x', encoding='utf-8') as file:
            json.dump(manifest, file, indent=2)
            file.write('\n')
        print(f'Saved {len(episode_rows)} episodes for spillover {spillover}.')
else:
    print('Reading the saved sensitivity results.')


## Compare paired policy runs

At spillover 0, neighbor harm is exactly zero, so its reward equals local reward; we reuse the saved local runs for both sides. At spillover 0.2, we reuse notebook 05's full-weight policies and the original local policies. A negative pollution gap means neighbor-harm reward lowered pollution. A positive cost gap means it spent more. Each gap pairs the same training seed.


In [ ]:
conditions = (
    (0.0, saved_zero_directory, saved_zero_directory, 'iql_local', 'iql_local'),
    (0.1, f'{output_directory}/spillover_10', f'{output_directory}/spillover_10', 'local', 'neighbor_harm'),
    (0.2, saved_original_directory, saved_neighbor_directory, 'iql_local', 'externality_10'),
    (0.4, f'{output_directory}/spillover_40', f'{output_directory}/spillover_40', 'local', 'neighbor_harm'),
)
paired_rows = []
policy_rows = []
for spillover, local_directory, harm_directory, local_name, harm_name in conditions:
    local_runs = pd.read_csv(f'{local_directory}/run_summary.csv')
    local_runs = local_runs[local_runs['policy'] == local_name].copy()
    harm_runs = pd.read_csv(f'{harm_directory}/run_summary.csv')
    harm_runs = harm_runs[harm_runs['policy'] == harm_name].copy()
    if set(local_runs['training_seed']) != set(training_seeds) or set(harm_runs['training_seed']) != set(training_seeds):
        raise ValueError(f'Training seeds differ at spillover {spillover}.')

    for label, runs in (('local', local_runs), ('neighbor_harm', harm_runs)):
        policy_rows.append({
            'spillover': spillover, 'policy': label,
            'pollution_mean': runs['mean_regional_pollution'].mean(),
            'pollution_sd': runs['mean_regional_pollution'].std(),
            'cost_mean': runs['mitigation_cost'].mean(),
            'cost_sd': runs['mitigation_cost'].std(),
            'threshold_mean': runs['threshold_exceedance'].mean(),
            'high_action_fraction_mean': runs['action_2_fraction'].mean(),
        })

    paired = harm_runs.merge(local_runs, on='training_seed', suffixes=('_harm', '_local'), validate='one_to_one')
    for _, row in paired.iterrows():
        paired_rows.append({
            'spillover': spillover, 'training_seed': int(row['training_seed']),
            'pollution_gap': row['mean_regional_pollution_harm'] - row['mean_regional_pollution_local'],
            'cost_gap': row['mitigation_cost_harm'] - row['mitigation_cost_local'],
            'threshold_gap': row['threshold_exceedance_harm'] - row['threshold_exceedance_local'],
        })

paired_results = pd.DataFrame(paired_rows)
policy_results = pd.DataFrame(policy_rows)
paired_summary = paired_results.groupby('spillover').agg(
    pollution_gap_mean=('pollution_gap', 'mean'),
    pollution_gap_sd=('pollution_gap', 'std'),
    cost_gap_mean=('cost_gap', 'mean'),
    cost_gap_sd=('cost_gap', 'std'),
    threshold_gap_mean=('threshold_gap', 'mean'),
    lower_pollution_runs=('pollution_gap', lambda values: (values < 0).sum()),
    higher_cost_runs=('cost_gap', lambda values: (values > 0).sum()),
    paired_runs=('training_seed', 'size'),
)
display(policy_results.round(3))
display(paired_summary.round(3))


## Pollution and cost together

The first figure gives mean paired changes with one standard deviation across training seeds. The second shows mean pollution and cost at each spillover strength. The units are synthetic, so there is no objective exchange rate between these outcomes. A policy is clearly better on these two metrics only if it lowers both.


In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4))
x = paired_summary.index.to_numpy()
axes[0].errorbar(x, paired_summary['pollution_gap_mean'],
                 yerr=paired_summary['pollution_gap_sd'], marker='o', capsize=4)
axes[0].axhline(0, color='gray', linewidth=1)
axes[0].set(xlabel='Spillover strength', ylabel='Neighbor harm minus local pollution')
axes[1].errorbar(x, paired_summary['cost_gap_mean'],
                 yerr=paired_summary['cost_gap_sd'], marker='o', capsize=4, color='tab:orange')
axes[1].axhline(0, color='gray', linewidth=1)
axes[1].set(xlabel='Spillover strength', ylabel='Neighbor harm minus local cost')
for axis in axes:
    axis.grid(alpha=0.2)
figure.tight_layout()
display(figure)
if write_comparison:
    figure.savefig(f'{output_directory}/paired_by_spillover.svg')
plt.close(figure)

figure, axis = plt.subplots(figsize=(8, 6))
for (spillover, policy), row in policy_results.set_index(['spillover', 'policy']).iterrows():
    color = 'tab:blue' if policy == 'local' else 'tab:orange'
    marker = 'o' if policy == 'local' else 's'
    axis.scatter(row['cost_mean'], row['pollution_mean'], color=color, marker=marker, s=60)
    axis.annotate(f'{spillover:.1f} {policy}', (row['cost_mean'], row['pollution_mean']),
                  xytext=(5, 4), textcoords='offset points', fontsize=8)
axis.set(xlabel='Mean mitigation cost', ylabel='Mean regional pollution')
axis.grid(alpha=0.2)
figure.tight_layout()
display(figure)
if write_comparison:
    figure.savefig(f'{output_directory}/pollution_vs_cost.svg')
plt.close(figure)


In [ ]:
if write_comparison:
    paired_results.to_csv(f'{output_directory}/paired_runs.csv', index=False)
    paired_summary.to_csv(f'{output_directory}/paired_summary.csv')
    policy_results.to_csv(f'{output_directory}/policy_comparison.csv', index=False)
    print('Saved comparison tables and figures.')
else:
    print('Saved comparison outputs remain unchanged.')


## Interpretation and conclusion

The full neighbor-harm reward did not reduce pollution at every spillover strength. At 0.1, its paired mean pollution was **0.11 higher** and mitigation cost **6.8 lower** than local reward; only 9 of 20 paired runs had lower pollution. At 0.2, pollution was **0.30 lower** and cost **50 higher**. At 0.4, pollution was **0.71 lower** and cost **73 higher**, with lower pollution in 17 of 20 runs. The fraction of region-steps above pollution 30 also fell by about 0.069 at spillover 0.4. At zero spillover the rewards are exactly identical, so their results are identical by construction.

**Takeaway:** Rewarding agents for reducing harm to neighbors looks more useful when spillover is stronger, but the pollution improvements came with higher mitigation cost. This is promising, not conclusive: the paired pollution changes varied substantially across training seeds (standard deviations of 1.04 at spillover 0.1 and 1.13 at 0.4), and four selected spillover values cannot establish a dependable trend or an optimal reward weight. Pollution and cost use arbitrary units, so the experiment cannot tell us whether the extra spending is worthwhile. Compare policies within each spillover condition, because changing physical spillover also changes baseline pollution.